# Step 1: Unstructured Knowledge Compilation & First-Gen 4. Wiki Agent (GCS)
## Agentic Ontology Studio Hands-on Series (Part 1/3 · Studio Tab 1~4)

- **Aligned Studio Tabs**: **`1. Asset` ➔ `2. OKF Layer` ➔ `3. Ontology Semantic` ➔ `4. Wiki Agent (First-Gen GCS Agent)`**
- **Learning Objective**: Overcome Vector RAG hallucinations by compiling unstructured policy guides into Google Cloud **Open Knowledge Format (OKF v0.2)** with clean **YAML Frontmatter (`---\n...\n---`)** and **Karpathy 3-Layer LLM-Wiki**, and launch the first-generation **`4. Wiki Agent` (GCS Agent, resolving 16/20 Policy CQs)** prior to any database graph setup.
- **Aligned Methodology**: [`Stage 2: Semantic Knowledge Modeling & OKF Specification`](../methodology/02_semantic_modeling_and_okf_en.md)
- **External Dependencies**: **0 (Pure Python Standard Library)**


### 2. Step 1 Architecture & Execution Flow (Flow Diagram)

![Step 1 Architecture Flow](attachment:step1_architecture_flow.png)


### [Exercise 1.1] Environment Setup & Loading Raw Policy Manual


In [1]:
import os, sys, json, hashlib

starter_root = os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
if starter_root not in sys.path:
    sys.path.insert(0, starter_root)

guide_path = os.path.join(starter_root, "data", "input", "01_guides", "sample_policy_guide.md")
with open(guide_path, "r", encoding="utf-8") as f:
    raw_content = f.read()

print(f"[OK] Python Runtime: {sys.version.split()[0]}")
print(f"[OK] Loaded Guide: {guide_path} ({len(raw_content)} chars)")
print("\n--- [Raw Policy Guide Preview] ---")
print(raw_content.strip())


[OK] Python Runtime: 3.13.7
[OK] Loaded Guide: /Users/seanjung/Downloads/Uplus Ontology/agentic-ontology-starter/data/input/01_guides/sample_policy_guide.md (1363 chars)

--- [Raw Policy Guide Preview] ---
# [Enterprise Service Consultation Guide: Terminal & Subscription Policy]
<!-- DOC_CHUNK_001: Overview & Scope -->
## 1. General Principles
This document defines standard commercial terms, device subsidy allocations, 180-day baseline plan commitment rules, and statutory difference penalty settlement policies for consumer subscriptions.

<!-- DOC_CHUNK_002: Baseline Plan Commitment (Six-Plan Rule) -->
## 2. 180-Day Baseline Plan Commitment Rule
1. Subscribers receiving public device subsidies must maintain their initial subscription tier at or above the designated Baseline Plan Threshold (e.g., 5G 45,000 KRW/month or LTE 26,400 KRW/month) for a mandatory period of **180 days** from the activation date.
2. If a subscriber downgrades below the baseline threshold within 180 days, a subsi

### [Exercise 1.2] Compiling OKF v0.2 & SHA-256 Hash Anchors (`pipeline.ingest`)


In [2]:
from pipeline.ingest import run_ingest

run_ingest(guide_path=guide_path, output_dir=os.path.join(starter_root, "data", "output"), wiki_dir=os.path.join(starter_root, "data", "llm_wiki"))

onto_path = os.path.join(starter_root, "data", "output", "ontology.jsonld")
with open(onto_path, "r", encoding="utf-8") as f:
    ontology = json.load(f)

print("=== [Google Cloud OKF v0.2 Bundle Metadata] ===")
print(f"@context: {ontology['@context']}")
print(f"Domain ID: {ontology['@graph'][0]['@id']}")
print(f"Total Chunks: {ontology['@graph'][0]['totalChunks']}")
print("\n=== [Anchored DocChunk Nodes] ===")
for node in ontology['@graph'][1:]:
    print(f" - ID: {node['@id']:<20} | Title: {node['title']:<35} | Hash Anchor: {node['hashAnchor']}")


[INGEST SUCCESS] Processed 3 chunks -> /Users/seanjung/Downloads/Uplus Ontology/agentic-ontology-starter/data/output/doc_chunks.jsonl, /Users/seanjung/Downloads/Uplus Ontology/agentic-ontology-starter/data/output/ontology.jsonld
[LLM-WIKI SUCCESS] Generated Karpathy 3-Layer Wiki in /Users/seanjung/Downloads/Uplus Ontology/agentic-ontology-starter/data/llm_wiki/
=== [Google Cloud OKF v0.2 Bundle Metadata] ===
@context: https://cloud.google.com/ontology/v0.2/context.jsonld
Domain ID: domain:TelecomSubsidyManagement
Total Chunks: 3

=== [Anchored DocChunk Nodes] ===
 - ID: chunk:DOC_CHUNK_001  | Title: Overview & Scope                    | Hash Anchor: anchor_a52e1a72869f
 - ID: chunk:DOC_CHUNK_002  | Title: Baseline Plan Commitment (Six-Plan Rule) | Hash Anchor: anchor_3003b683cea0
 - ID: chunk:DOC_CHUNK_003  | Title: Penalty Math Formula                | Hash Anchor: anchor_47971ec8e4d8


### [Exercise 1.3] Validating Karpathy 3-Layer LLM-Wiki Layers


In [3]:
wiki_dir = os.path.join(starter_root, "data", "llm_wiki")
wiki_files = ["summary.md", "entities.md", "concepts.md", "rules.md"]

print("=== [Karpathy 3-Layer LLM-Wiki Verification] ===")
for wf in wiki_files:
    p = os.path.join(wiki_dir, wf)
    sz = os.path.getsize(p)
    with open(p, "r", encoding="utf-8") as f:
        first_line = f.readline().strip()
    print(f" - Layer File: {wf:<15} ({sz:>5} bytes) -> {first_line}")

print("\n=== [Layer 3: Mathematical Rules Excerpt from rules.md] ===")
with open(os.path.join(wiki_dir, "rules.md"), "r", encoding="utf-8") as f:
    r_content = f.read()
for line in r_content.splitlines():
    if "Penalty" in line or "180" in line or "Exemption" in line:
        print(f"  > {line}")


=== [Karpathy 3-Layer LLM-Wiki Verification] ===
 - Layer File: summary.md      (  561 bytes) -> # Enterprise Knowledge Wiki: Executive Summary (Layer 1)
 - Layer File: entities.md     (  831 bytes) -> # Enterprise Knowledge Wiki: Candidate Entities Directory (Layer 2)
 - Layer File: concepts.md     (  804 bytes) -> # Enterprise Knowledge Wiki: 78 Semantic Concepts & Taxonomies (Layer 3)
 - Layer File: rules.md        (  956 bytes) -> # Enterprise Knowledge Wiki: Codified Business Rules (Layer 3)

=== [Layer 3: Mathematical Rules Excerpt from rules.md] ===
  > ### Rule 1: 180-Day Six-Plan Commitment Obligation (`RULE_SIX_PLAN_180D`)
  > - **Constraint**: Must maintain baseline tier or higher for at least 180 days from activation date.
  > - **Penalty on Breach (< 180 Days)**:
  >   `Penalty = (Initial Subsidy - Downgraded Plan Subsidy) * (180 - Maintained Days) / 180`
  > - **Post-180 Days**: Differential clawback is completely waived (0 KRW penalty).


### [Exercise 1.4] Executing Hash-Grounded Citation Agent


In [4]:
chunks_path = os.path.join(starter_root, "data", "output", "doc_chunks.jsonl")
chunks = []
with open(chunks_path, "r", encoding="utf-8") as f:
    for line in f:
        chunks.append(json.loads(line))

def retrieve_policy_with_anchor(keyword: str):
    matched = [c for c in chunks if keyword in c["content"] or keyword in c["title"]]
    return [{
        "chunk_id": c["chunk_id"],
        "title": c["title"],
        "hash_anchor": c["hash_anchor"],
        "content_snippet": c["content"].replace("\n", " ")[:150] + "..."
    } for c in matched]

print("=== [Agent Retrieval Grounding Test: '180 days commitment'] ===")
results = retrieve_policy_with_anchor("180")
for item in results:
    print(f"✔ [Verified Grounding] Chunk: {item['chunk_id']} | Hash Anchor: {item['hash_anchor']}")
    print(f"  Title: {item['title']}")
    print(f"  Snippet: {item['content_snippet']}\n")


=== [Agent Retrieval Grounding Test: '180 days commitment'] ===
✔ [Verified Grounding] Chunk: DOC_CHUNK_001 | Hash Anchor: anchor_a52e1a72869f
  Title: Overview & Scope
  Snippet: ## 1. General Principles This document defines standard commercial terms, device subsidy allocations, 180-day baseline plan commitment rules, and stat...

✔ [Verified Grounding] Chunk: DOC_CHUNK_002 | Hash Anchor: anchor_3003b683cea0
  Title: Baseline Plan Commitment (Six-Plan Rule)
  Snippet: ## 2. 180-Day Baseline Plan Commitment Rule 1. Subscribers receiving public device subsidies must maintain their initial subscription tier at or above...

✔ [Verified Grounding] Chunk: DOC_CHUNK_003 | Hash Anchor: anchor_47971ec8e4d8
  Title: Penalty Math Formula
  Snippet: ## 3. Penalty Settlement Computation Formula The subsidy difference penalty is calculated according to the following deterministic formula: `Penalty =...



### 3. Key Findings & Practical Interpretation (Interpretation)
- **Hash Immutability**: Any modification to a single character in the policy manual instantly alters the `hash_anchor`, providing built-in tamper detection.
- **Cognitive Isolation**: Instead of loading the entire 50-page manual, the LLM reads `summary.md` -> `rules.md`, saving tokens and eliminating hallucination.
- **Handoff to Step 2**: These anchored `DocChunk` and `DomainRule` nodes will be directly linked with BigQuery DW master tables in Step 2 via Semantic Bridges.
